# Notebook 2: Condicionales y Bifurcaciones en LangGraph

## ¿De qué trata este notebook?

En el Notebook 1 todos los grafos eran lineales: A → B → C, sin posibilidad de elegir.
En la realidad, los sistemas de IA necesitan **tomar decisiones**: según lo que reciban, ejecutan un camino u otro.

### La analogía del aeropuerto

Imagina el control de seguridad de un aeropuerto. Hay un escáner inicial (clasificador) que decide a qué cola va cada pasajero:
- Pasajero con maleta grande → cola larga
- Pasajero sin maleta → cola rápida
- Pasajero con artículos especiales → revisión manual

LangGraph hace exactamente lo mismo con los mensajes: un nodo clasificador decide qué camino tomar.

### El flujo que construiremos

```
                  ┌─── nodo_matematicas ───┐
START → clasificar─┼─── nodo_cultura      ──┼→ END
                  └─── nodo_codigo       ───┘
```

### Conceptos nuevos en este notebook

| Concepto | ¿Qué es? |
|----------|---------|
| **Conditional edge** | Una flecha que va a sitios diferentes según una condición |
| **Función de routing** | La función que decide a qué nodo ir |
| **path_map** | El mapa que relaciona valores con nombres de nodos |
| **Branches** | Las ramas del grafo (los caminos alternativos) |

---
## 1. Instalación y configuración

In [ ]:
# %pip install -qU langgraph langchain-google-genai

In [4]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="C:/Users/Oscar/OneDrive - FM4/Escritorio/EVOLVE/Data Science/EVOLVE/Victor_Prior_IA_Generativa/Proyecto_Modulo_IAGen/.env")
API_KEY = os.getenv("GEMINI_API_KEY_001")

# API key de Gemini
# API_KEY = userdata.get('GEMINI_API_KEY')

In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3,  # Más determinista para clasificación
    api_key = API_KEY

)
print("Modelo listo:", llm.model)

Modelo listo: gemini-2.5-flash


---
## 2. El problema: diferentes preguntas necesitan diferentes respuestas

Imaginemos un asistente que recibe preguntas de tres tipos:
- **Matemáticas**: necesita un nodo especializado que razone paso a paso
- **Cultura general**: respuesta directa con el LLM
- **Código**: responde con un ejemplo de código

El flujo que vamos a construir:

```
                  ┌─── nodo_matematicas ───┐
START → clasificar─┼─── nodo_cultura      ──┼→ END
                  └─── nodo_codigo       ───┘
```

---
## 3. Definir el Estado

Este estado tiene un campo nuevo: `tipo`. El nodo clasificador lo rellenará con la categoría detectada, y ese valor es el que usará la función de routing para decidir a qué nodo ir.

`nodo_usado` sirve como registro de auditoría: al final sabremos exactamente qué rama tomó el grafo.

In [8]:
from typing import TypedDict, Literal

class EstadoAsistente(TypedDict):
    pregunta: str
    tipo: str           # 'matematicas', 'cultura', 'codigo'
    respuesta: str
    nodo_usado: str     # Para saber qué rama tomó el grafo

---
## 4. Nodo clasificador

### ¿Qué hace este nodo?

Este nodo es el escáner del aeropuerto: examina la pregunta y decide a qué cola debe ir.

Le pide al LLM que responda **solo con una palabra** de una lista fija. Eso hace la respuesta predecible y fácil de procesar.

**Punto crítico:** si el LLM responde algo inesperado, el código lo manda a `cultura` como fallback seguro.

**Lo que entra:** `pregunta` del estado  
**Lo que sale:** campo `tipo` rellenado con `matematicas`, `codigo` o `cultura`

In [9]:
def nodo_clasificar(estado: EstadoAsistente) -> dict:
    """
    Clasifica la pregunta del usuario en una de tres categorías.
    """
    prompt = f"""Clasifica la siguiente pregunta en UNA de estas categorías:
- matematicas (si involucra cálculos, álgebra, estadística, lógica numérica)
- codigo (si pide escribir, debuggear o explicar código)
- cultura (cualquier otra pregunta de conocimiento general)

Responde SOLO con una de estas palabras: matematicas, codigo, cultura

Pregunta: {estado['pregunta']}"""

    respuesta = llm.invoke(prompt)
    tipo = respuesta.content.strip().lower()
    
    # Normalizar en caso de respuesta inesperada
    if tipo not in ["matematicas", "codigo", "cultura"]:
        tipo = "cultura"
    
    print(f"[Clasificador] Pregunta: '{estado['pregunta'][:50]}' → Tipo: '{tipo}'")
    return {"tipo": tipo}

---
## 5. Nodos especializados (las ramas)

### ¿Por qué nodos separados para cada tipo?

Cada tipo de pregunta se beneficia de un prompt distinto:
- El nodo de **matemáticas** pide razonar paso a paso
- El nodo de **cultura** pide respuestas concisas
- El nodo de **código** pide siempre incluir un ejemplo ejecutable

Todos devuelven `respuesta` y `nodo_usado` (auditoría de qué camino tomó el grafo).

In [10]:
def nodo_matematicas(estado: EstadoAsistente) -> dict:
    """
    Resuelve preguntas matemáticas paso a paso.
    """
    prompt = f"""Eres un tutor de matemáticas. Resuelve la siguiente pregunta mostrando
cada paso claramente. Usa notación clara.

Pregunta: {estado['pregunta']}"""
    
    respuesta = llm.invoke(prompt)
    print("[Nodo Matemáticas] Procesando...")
    return {
        "respuesta": respuesta.content,
        "nodo_usado": "matematicas"
    }


def nodo_cultura(estado: EstadoAsistente) -> dict:
    """
    Responde preguntas de cultura general de forma concisa.
    """
    prompt = f"""Eres un asistente enciclopédico. Responde de forma concisa y clara.

Pregunta: {estado['pregunta']}"""
    
    respuesta = llm.invoke(prompt)
    print("[Nodo Cultura] Procesando...")
    return {
        "respuesta": respuesta.content,
        "nodo_usado": "cultura"
    }


def nodo_codigo(estado: EstadoAsistente) -> dict:
    """
    Ayuda con preguntas de programación, siempre con ejemplo de código.
    """
    prompt = f"""Eres un experto en programación. Responde siempre incluyendo un ejemplo
de código funcional. Usa bloques de código con el lenguaje apropiado.

Pregunta: {estado['pregunta']}"""
    
    respuesta = llm.invoke(prompt)
    print("[Nodo Código] Procesando...")
    return {
        "respuesta": respuesta.content,
        "nodo_usado": "codigo"
    }

---
## 6. La función de routing (enrutamiento)

### ¿Qué es la función de routing?

Esta es la pieza clave de los conditional edges. Es una función simple que:
- Recibe el estado actual
- Lee el campo `tipo` (que el clasificador ya rellenó)
- Devuelve un string con el nombre del nodo al que debe ir el flujo

LangGraph lee ese string y envía la ejecución al nodo correspondiente.

**El `Literal[...]`** le dice al código que esta función solo puede devolver uno de esos tres valores. Es una declaración de intenciones que ayuda a detectar errores antes de ejecutar.

In [11]:
def router(estado: EstadoAsistente) -> Literal["matematicas", "cultura", "codigo"]:
    """
    Función de routing: lee el tipo del estado y retorna el nombre del nodo destino.
    El valor retornado debe corresponder a un nombre de nodo registrado en el grafo.
    """
    tipo = estado["tipo"]
    print(f"[Router] Dirigiendo al nodo: '{tipo}'")
    return tipo  # 'matematicas', 'cultura' o 'codigo'

---
## 7. Construir el grafo con conditional edges

### ¿Cómo se añade una bifurcación?

Con `add_conditional_edges()` en lugar de `add_edge()`. Parámetros:
- `source`: el nodo desde donde sale la bifurcación
- `path`: la función de routing que decide a dónde ir
- `path_map`: diccionario `{valor_posible: nodo_destino}`

Al final, los tres nodos especializados van todos al `END` con `add_edge` normal.

In [12]:
from langgraph.graph import StateGraph, START, END

grafo = StateGraph(EstadoAsistente)

# Agregar todos los nodos
grafo.add_node("clasificar",   nodo_clasificar)
grafo.add_node("matematicas",  nodo_matematicas)
grafo.add_node("cultura",      nodo_cultura)
grafo.add_node("codigo",       nodo_codigo)

# Edge fijo: siempre empieza en 'clasificar'
grafo.add_edge(START, "clasificar")

# Edge condicional: DESPUÉS de 'clasificar', el router decide a dónde ir
grafo.add_conditional_edges(
    source="clasificar",     # Nodo de origen
    path=router,             # Función que decide el destino
    path_map={               # Mapa de valores posibles → nodos destino
        "matematicas": "matematicas",
        "cultura":     "cultura",
        "codigo":      "codigo",
    }
)

# Todos los nodos especializados van al END
grafo.add_edge("matematicas", END)
grafo.add_edge("cultura",     END)
grafo.add_edge("codigo",      END)

app = grafo.compile()
print("✅ Grafo con bifurcaciones compilado")

✅ Grafo con bifurcaciones compilado


In [13]:
# Visualizar el grafo (notarás que ahora tiene múltiples caminos)
print(app.get_graph().draw_ascii())

                    +-----------+                         
                    | __start__ |                         
                    +-----------+                         
                           *                              
                           *                              
                           *                              
                    +------------+                        
                    | clasificar |                        
                  ..+------------+..                      
               ...         .        ....                  
           ....           .             ....              
         ..               .                 ..            
+--------+          +---------+          +-------------+  
| codigo |*         | cultura |          | matematicas |  
+--------+ ****     +---------+         *+-------------+  
               ***        *         ****                  
                  ****     *    ****                    

---
## 8. Probar el grafo con diferentes preguntas

`ejecutar_asistente` es una función auxiliar de presentación. Ejecuta el grafo y muestra los resultados de forma limpia. No forma parte del grafo en sí.

In [14]:
def ejecutar_asistente(pregunta: str):
    """Helper para ejecutar el asistente y mostrar el resultado."""
    print("\n" + "=" * 65)
    print(f"PREGUNTA: {pregunta}")
    print("=" * 65)
    
    estado_inicial = {
        "pregunta": pregunta,
        "tipo": "",
        "respuesta": "",
        "nodo_usado": ""
    }
    
    resultado = app.invoke(estado_inicial)
    
    print(f"\n✅ Nodo usado: '{resultado['nodo_usado']}'")
    print(f"\nRESPUESTA:\n{resultado['respuesta']}")
    return resultado

In [15]:
# Prueba 1: Pregunta matemática
r1 = ejecutar_asistente("¿Cuánto es la raíz cuadrada de 144 más el 15% de 200?")


PREGUNTA: ¿Cuánto es la raíz cuadrada de 144 más el 15% de 200?
[Clasificador] Pregunta: '¿Cuánto es la raíz cuadrada de 144 más el 15% de 2' → Tipo: 'matematicas'
[Router] Dirigiendo al nodo: 'matematicas'
[Nodo Matemáticas] Procesando...

✅ Nodo usado: 'matematicas'

RESPUESTA:
¡Hola! ¡Claro que sí! Vamos a resolver este problema paso a paso para que quede todo muy claro.

La pregunta es: **¿Cuánto es la raíz cuadrada de 144 más el 15% de 200?**

Vamos a dividir el problema en dos partes y luego sumaremos los resultados.

---

**Paso 1: Calcular la raíz cuadrada de 144**

*   **Concepto:** La raíz cuadrada de un número es otro número que, multiplicado por sí mismo, da el número original.
*   **Cálculo:** Buscamos un número 'x' tal que $x \times x = 144$.
    *   Sabemos que $10 \times 10 = 100$
    *   Sabemos que $11 \times 11 = 121$
    *   Sabemos que $12 \times 12 = 144$
*   **Resultado del Paso 1:**
    $\sqrt{144} = 12$

---

**Paso 2: Calcular el 15% de 200**

*   **Concepto:

In [16]:
# Prueba 2: Pregunta de cultura
r2 = ejecutar_asistente("¿En qué año cayó el muro de Berlín?")


PREGUNTA: ¿En qué año cayó el muro de Berlín?
[Clasificador] Pregunta: '¿En qué año cayó el muro de Berlín?' → Tipo: 'cultura'
[Router] Dirigiendo al nodo: 'cultura'
[Nodo Cultura] Procesando...

✅ Nodo usado: 'cultura'

RESPUESTA:
El Muro de Berlín cayó en 1989.


In [17]:
# Prueba 3: Pregunta de código
r3 = ejecutar_asistente("¿Cómo se hace un decorador en Python?")


PREGUNTA: ¿Cómo se hace un decorador en Python?
[Clasificador] Pregunta: '¿Cómo se hace un decorador en Python?' → Tipo: 'codigo'
[Router] Dirigiendo al nodo: 'codigo'
[Nodo Código] Procesando...

✅ Nodo usado: 'codigo'

RESPUESTA:
¡Absolutamente! Como experto en programación, te guiaré a través de la creación de decoradores en Python, una herramienta poderosa para extender o modificar el comportamiento de funciones o clases sin alterar su código fuente.

### ¿Qué es un Decorador en Python?

En su esencia, un **decorador** es una función que toma otra función como argumento, le añade alguna funcionalidad y luego devuelve la función modificada. Piensa en ello como envolver un regalo: el regalo (tu función original) sigue siendo el mismo, pero le añades un envoltorio bonito (la funcionalidad extra del decorador).

Los decoradores son muy útiles para tareas como:
*   **Logging**: Registrar llamadas a funciones, argumentos y resultados.
*   **Medición de tiempo**: Calcular cuánto tarda un

---
## 9. Variante: routing sin path_map

Cuando los valores que devuelve el router son exactamente los mismos nombres que los nodos, puedes omitir `path_map`. LangGraph usará el valor devuelto directamente como nombre del nodo.

| Opción | Cuándo usarla |
|--------|--------------|
| Con `path_map` | Los valores del router son distintos a los nombres de nodos |
| Sin `path_map` | El router devuelve exactamente los nombres de los nodos (más limpio) |

In [18]:
# Mismo resultado, sintaxis más corta (cuando los valores son exactamente los nombres de nodo)
grafo2 = StateGraph(EstadoAsistente)
grafo2.add_node("clasificar",  nodo_clasificar)
grafo2.add_node("matematicas", nodo_matematicas)
grafo2.add_node("cultura",     nodo_cultura)
grafo2.add_node("codigo",      nodo_codigo)
grafo2.add_edge(START, "clasificar")

# Sin path_map: la función retorna el nombre del nodo directamente
grafo2.add_conditional_edges("clasificar", router)

grafo2.add_edge("matematicas", END)
grafo2.add_edge("cultura", END)
grafo2.add_edge("codigo", END)

app2 = grafo2.compile()
print("✅ Grafo alternativo (sin path_map) compilado")

✅ Grafo alternativo (sin path_map) compilado


---
## 10. Routing hacia END directamente

El router puede terminar el grafo devolviendo `END` directamente, sin pasar por otro nodo.

**Cuándo usar esto:** cuando la entrada no cumple una condición mínima (está vacía, es inválida...) y no tiene sentido continuar procesando.

In [19]:
from langgraph.graph import END

# Ejemplo: si la pregunta está vacía, terminar sin procesar
def router_con_guard(estado: EstadoAsistente) -> str:
    if not estado["pregunta"].strip():
        print("[Router] Pregunta vacía, terminando.")
        return END  # Se puede retornar END directamente
    return estado["tipo"]

print("Router con guard definido")

Router con guard definido


---
## 11. Ejercicios propuestos

1. **Añade una cuarta categoría** `historia` con un nodo especializado que siempre responda dando el contexto histórico completo.

2. **Dos clasificaciones en cascada**: primero clasifica si la pregunta es "técnica" o "no técnica", y si es técnica, clasifica entre "matemáticas" y "código".

3. **Nodo de fallback**: añade un nodo `desconocido` al que vaya el router cuando la pregunta no encaja en ninguna categoría conocida.

## Resumen

| Concepto | Descripción |
|---|---|
| `add_conditional_edges` | Conecta un nodo a múltiples destinos posibles |
| Función de routing | Recibe el estado, retorna el nombre del siguiente nodo |
| `path_map` | Diccionario que mapea valores retornados a nombres de nodo |
| Retornar `END` | El router puede terminar el grafo directamente |
| Branches independientes | Cada rama puede tener su propia lógica especializada |